In [1]:
import sys
import os
sys.path
sys.path.append('..')
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Subset
from torch.utils.data.dataset import ConcatDataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# Import your model and dataset class
from utilities import EfficientNetMultiOutput_v4
from utilities import PreprocessedDataset
from utilities import UnfreezeCallbackEffNet

In [3]:
preprocessed_dir = '../data/preprocessed_EDA'
cleaned_csv = "../data/cleaned_data.csv"
df = pd.read_csv(cleaned_csv)

In [4]:
df.sample(5)

,filename,age,gender
3789,23_1_2_20170116173017785.jpg.chip.jpg,23,1
2281,1_1_2_20161219155353413.jpg.chip.jpg,1,1
20380,66_1_0_20170110141136857.jpg.chip.jpg,66,1
20555,68_1_0_20170103183639034.jpg.chip.jpg,68,1
11303,32_0_3_20170119200432788.jpg.chip.jpg,32,0


In [5]:
n_splits = 5
batch_size = 32
num_epochs = 50

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from pytorch_lightning import Trainer, loggers, callbacks

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['gender'])):
    print(f"\n🔹 Fold {fold + 1}/{n_splits}")

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = PreprocessedDataset(train_df, preprocessed_dir)
    val_dataset = PreprocessedDataset(val_df, preprocessed_dir)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, persistent_workers=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True, persistent_workers=True
    )

    # Compute class weights for gender
    gender_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(2),
        y=train_df['gender']
    )
    gender_weights = torch.tensor(gender_weights, dtype=torch.float32)
    class_weights = {'gender': gender_weights}

    # Logger
    logger = loggers.TensorBoardLogger(f'../tb_logs', name=f'EffNet_regression_fold{fold + 1}')

    # Callbacks
    checkpoint_callback = callbacks.ModelCheckpoint(
        monitor='val_loss',
        filename=f'fold{fold + 1}-best-{{epoch:02d}}-{{val_loss:.2f}}',
        save_top_k=1,
        mode='min',
        dirpath='../models/'
    )
    early_stop_callback = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        mode='min'
    )
    unfreeze_callback = callbacks.BackboneFinetuning(
        unfreeze_backbone_at_epoch=7,
        backbone_initial_ratio_lr=0.1
    )

    # Initialize model (IMPORTANT: make sure the model’s backbone is named `.backbone`)
    model = EfficientNetMultiOutput_v4(class_weights=class_weights)

    # Setup Trainer
    trainer = Trainer(
        max_epochs=num_epochs,
        accelerator='gpu',
        devices=1,
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback, unfreeze_callback],
        precision='16-mixed',
        accumulate_grad_batches=4,
        log_every_n_steps=10
    )

    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

    print(f"✅ Fold {fold + 1} completed!")


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision



🔹 Fold 1/5


C:\Users\MiestoMeska\anaconda3\envs\TC_4\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:654: Checkpoint directory C:\Users\MiestoMeska\Desktop\Mokslai\VIKO\BD\CNN_AgeGenderRace\models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\MiestoMeska\anaconda3\envs\TC_4\lib\site-packages\torch\optim\lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "

  | Name              | Type                | Params | Mode 
------------------------------------------------------------------
0 | backbone          | EfficientNet        | 4.0 M  | train
1 | gender_classifier | Sequential          | 2.0 M  | train
2 | age_regressor     | Sequential          | 2.0 M  | train
3 | criterion_gender  | CrossEntropyLoss    | 0      | train
4 | criterion_age     | MSELoss             | 0      | train
5 | gender_accuracy 

Sanity Checking: |                                                                               | 0/? [00:00<…

Training: |                                                                                      | 0/? [00:00<…

...

`Trainer.fit` stopped: `max_epochs=50` reached.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
C:\Users\MiestoMeska\anaconda3\envs\TC_4\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:654: Checkpoint directory C:\Users\MiestoMeska\Desktop\Mokslai\VIKO\BD\CNN_AgeGenderRace\models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\MiestoMeska\anaconda3\envs\TC_4\lib\site-packages\torch\optim\lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "

  | Name              | Type                | Params | Mode 
------------------------------------------------------------------
0 | backbone          | EfficientNet        | 4.0 M  | train
1 | gender_classifier | Sequential          | 2.0 M  | t

✅ Fold 1 completed!

🔹 Fold 2/5


Sanity Checking: |                                                                               | 0/? [00:00<…

Training: |                                                                                      | 0/? [00:00<…


Detected KeyboardInterrupt, attempting graceful shutdown ...
